In [0]:
import logging
from src.config_loader import load_config

from src.manifest import (
    get_manifest_table_name,
    get_latest_manifest_state,
    get_latest_successful_manifest,
    append_manifest_records
)

from src.bls_ingestion import (
    fetch_bls_directory,
    parse_bls_inventory,
    build_comparison_df,
    download_bls_files,
    detect_removed_files,
    build_removed_manifest_records
)

from src.population_ingestion import (
    fetch_population,
    write_population_file
)


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    force=True
)

logger = logging.getLogger(__name__)

In [0]:
config = load_config(
    "../configs/config.yaml"
)

manifest_table = (
    get_manifest_table_name(config)
)

bls_config = config["sources"]["bls"]

bls_headers = {
    "User-Agent": bls_config["user_agent"]
}

logger.info(
    "Starting BLS ingestion. Manifest table=%s",
    manifest_table
)

bls_html = fetch_bls_directory(
    bls_config["url"],
    bls_headers
)

bls_inventory = parse_bls_inventory(
    bls_html,
    bls_config["url"]
)

if not bls_inventory:
    raise ValueError(
        "BLS inventory is empty. "
        "Source directory parsing may have failed."
    )

logger.info(
    "BLS inventory created with %s file(s).",
    len(bls_inventory)
)

inventory_df = spark.createDataFrame(
    bls_inventory
)

latest_bls_success_df = (
    get_latest_successful_manifest(
        spark,
        manifest_table,
        "BLS"
    )
)

latest_bls_state_df = (
    get_latest_manifest_state(
        spark,
        manifest_table,
        "BLS"
    )
)

comparison_df = build_comparison_df(
    spark,
    bls_inventory,
    latest_bls_success_df
)

action_counts = {
    row["action"]: row["count"]
    for row in (
        comparison_df
        .groupBy("action")
        .count()
        .collect()
    )
}

logger.info(
    "BLS comparison result: %s",
    action_counts
)

bls_results = download_bls_files(
    comparison_df,
    bls_config["target_path"],
    bls_headers
)

append_manifest_records(
    spark,
    manifest_table,
    bls_results
)

removed_files_df = detect_removed_files(
    latest_bls_state_df,
    inventory_df
)

removed_results = build_removed_manifest_records(
    removed_files_df
)

append_manifest_records(
    spark,
    manifest_table,
    removed_results
)

logger.info(
    "BLS ingestion completed. Download records=%s Removed records=%s",
    len(bls_results),
    len(removed_results)
)

In [0]:
population_config = (
    config["sources"]["population"]
)

logger.info(
    "Starting Population API ingestion."
)

population_response = fetch_population(
    population_config["url"]
)

population_action, population_results = (
    write_population_file(
        population_response,
        population_config["target_path"]
    )
)

logger.info(
    "Population ingestion action=%s",
    population_action
)

append_manifest_records(
    spark,
    manifest_table,
    population_results
)

if population_results:
    logger.info(
        "Population manifest record inserted."
    )
else:
    logger.info(
        "No new Population manifest record required."
    )

logger.info(
    "Population ingestion completed."
)